In [20]:
import numpy as np
import pandas as pd
import os

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
from tensorflow.keras.preprocessing.image import load_img, img_to_array

import cv2


In [21]:
df = pd.read_csv('image_metadata.csv')
df

,file_path,experiment_id,position,trap_num,sample_id,treated,timepoint,species_label
0,C:\Users\silve\Documents\210\data\Cropped-png-...,BV6126 AST FISH 201116,101,16,43-16,Untreated,0,P. aeruginosa
1,C:\Users\silve\Documents\210\data\Cropped-png-...,BV6126 AST FISH 201116,101,16,43-16,Untreated,1,P. aeruginosa
2,C:\Users\silve\Documents\210\data\Cropped-png-...,BV6126 AST FISH 201116,101,16,43-16,Untreated,2,P. aeruginosa
3,C:\Users\silve\Documents\210\data\Cropped-png-...,BV6126 AST FISH 201116,101,16,43-16,Untreated,3,P. aeruginosa
4,C:\Users\silve\Documents\210\data\Cropped-png-...,BV6126 AST FISH 201116,101,16,43-16,Untreated,4,P. aeruginosa
...,...,...,...,...,...,...,...,...
106838,C:\Users\silve\Documents\210\data\Cropped-png-...,BV6134 AST FISH 201126,281,20,3408-03,Treated,25,E. coli
106839,C:\Users\silve\Documents\210\data\Cropped-png-...,BV6134 AST FISH 201126,281,20,3408-03,Treated,26,E. coli
106840,C:\Users\silve\Documents\210\data\Cropped-png-...,BV6134 AST FISH 201126,281,20,3408-03,Treated,27,E. coli
106841,C:\Users\silve\Documents\210\data\Cropped-png-...,BV6134 AST FISH 201126,281,20,3408-03,Treated,28,E. coli


In [24]:
filtered_df = df[(df['species_label'] != 'Unknown') & #only working with known species
                 (df['treated'] == 'Untreated') &  # removed treated antibiotic samples since the later timepoint images are harder to use
                 (df['timepoint'] == 15)] # looking at frame 15 for the 30 minute point
filtered_df

,file_path,experiment_id,position,trap_num,sample_id,treated,timepoint,species_label
15,C:\Users\silve\Documents\210\data\Cropped-png-...,BV6126 AST FISH 201116,101,16,43-16,Untreated,15,P. aeruginosa
49,C:\Users\silve\Documents\210\data\Cropped-png-...,BV6126 AST FISH 201116,101,30,43-08,Untreated,15,P. aeruginosa
83,C:\Users\silve\Documents\210\data\Cropped-png-...,BV6126 AST FISH 201116,101,40,43-18,Untreated,15,P. aeruginosa
117,C:\Users\silve\Documents\210\data\Cropped-png-...,BV6126 AST FISH 201116,101,41,43-19,Untreated,15,P. aeruginosa
151,C:\Users\silve\Documents\210\data\Cropped-png-...,BV6126 AST FISH 201116,102,7,123-07,Untreated,15,P. aeruginosa
...,...,...,...,...,...,...,...,...
100708,C:\Users\silve\Documents\210\data\Cropped-png-...,BV6134 AST FISH 201126,179,29,5647-12,Untreated,15,K. pneumoniae
100738,C:\Users\silve\Documents\210\data\Cropped-png-...,BV6134 AST FISH 201126,180,23,5715-06,Untreated,15,P. aeruginosa
100768,C:\Users\silve\Documents\210\data\Cropped-png-...,BV6134 AST FISH 201126,180,25,5715-08,Untreated,15,K. pneumoniae
100798,C:\Users\silve\Documents\210\data\Cropped-png-...,BV6134 AST FISH 201126,180,29,5715-12,Untreated,15,K. pneumoniae


In [25]:
filtered_df['species_label'].value_counts()

species_label
K. pneumoniae    1271
E. coli           582
E. faecalis       371
P. aeruginosa     250
Name: count, dtype: int64

In [26]:
n = 250
filtered_df_sampled = filtered_df.groupby('species_label').apply(lambda x: x.sample(n=n, random_state=18)).reset_index(drop=True)
filtered_df_sampled #adjusted to 250 samples per species to account for majority class imbalance

C:\Users\silve\AppData\Local\Temp\ipykernel_22808\1369163131.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  filtered_df_sampled = filtered_df.groupby('species_label').apply(lambda x: x.sample(n=n, random_state=18)).reset_index(drop=True)


,file_path,experiment_id,position,trap_num,sample_id,treated,timepoint,species_label
0,C:\Users\silve\Documents\210\data\Cropped-png-...,BV6129 AST FISH 201117,111,30,1125-11,Untreated,15,E. coli
1,C:\Users\silve\Documents\210\data\Cropped-png-...,BV6134 AST FISH 201126,134,3,2587-03,Untreated,15,E. coli
2,C:\Users\silve\Documents\210\data\Cropped-png-...,BV6128 AST FISH 201117,174,2,5647-02,Untreated,15,E. coli
3,C:\Users\silve\Documents\210\data\Cropped-png-...,BV6129 AST FISH 201117,150,13,3777-13,Untreated,15,E. coli
4,C:\Users\silve\Documents\210\data\Cropped-png-...,BV6129 AST FISH 201117,140,23,3097-04,Untreated,15,E. coli
...,...,...,...,...,...,...,...,...
995,C:\Users\silve\Documents\210\data\Cropped-png-...,BV6131 AST FISH 201124,133,15,2803-15,Untreated,15,P. aeruginosa
996,C:\Users\silve\Documents\210\data\Cropped-png-...,BV6126 AST FISH 201116,186,14,6843-14,Untreated,15,P. aeruginosa
997,C:\Users\silve\Documents\210\data\Cropped-png-...,BV6127 AST FISH 201117,138,22,2791-05,Untreated,15,P. aeruginosa
998,C:\Users\silve\Documents\210\data\Cropped-png-...,BV6126 AST FISH 201116,146,37,3643-15,Untreated,15,P. aeruginosa


failed attempt 1

In [67]:
def load_image(file_path, target_size=(64, 64)):
    img = load_img(file_path, target_size=target_size)
    img = img_to_array(img)
    img = img / 255.0  # Normalize the image
    return img

In [68]:
X = []
y = []

for _, row in filtered_df_sampled.iterrows():
    img_path = row['file_path']  # path to image file
    species_label = row['species_label']  # target variable
    
    image = load_image(img_path)
    X.append(image)
    
    species_map = {'E. faecalis': 0, 'K. pneumoniae': 1, 'E. coli': 2, 'P. aeruginosa': 3}
    y.append(species_map.get(species_label, -1))

In [69]:
X = np.array(X)
y = np.array(y)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=18, stratify=y)

In [70]:
print(f"Training set class distribution: {pd.Series(y_train).value_counts()}")
print(f"Test set class distribution: {pd.Series(y_test).value_counts()}")

Training set class distribution: 3    160
0    160
1    160
2    160
Name: count, dtype: int64
Test set class distribution: 0    40
2    40
1    40
3    40
Name: count, dtype: int64


In [71]:
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(64, 64, 3)),
    MaxPooling2D(pool_size=(2, 2)),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D(pool_size=(2, 2)),
    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D(pool_size=(2, 2)),
    Flatten(),
    Dense(128, activation='relu'),
    Dense(4, activation='softmax')  # 4 species classes
])


model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

c:\Users\silve\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [72]:
model.fit(X_train, y_train, epochs=10, batch_size=32, validation_data=(X_test, y_test))

Epoch 1/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 3s 40ms/step - accuracy: 0.2264 - loss: 1.5571 - val_accuracy: 0.2500 - val_loss: 1.3871
Epoch 2/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 31ms/step - accuracy: 0.2562 - loss: 1.3874 - val_accuracy: 0.2500 - val_loss: 1.3865
Epoch 3/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 31ms/step - accuracy: 0.2823 - loss: 1.3851 - val_accuracy: 0.2500 - val_loss: 1.3863
Epoch 4/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 32ms/step - accuracy: 0.2405 - loss: 1.3866 - val_accuracy: 0.2500 - val_loss: 1.3863
Epoch 5/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 30ms/step - accuracy: 0.2461 - loss: 1.3863 - val_accuracy: 0.2500 - val_loss: 1.3863
Epoch 6/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 33ms/step - accuracy: 0.2146 - loss: 1.3864 - val_accuracy: 0.2500 - val_loss: 1.3863
Epoch 7/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 31ms/step - accuracy: 0.2557 - loss: 1.3863 - val_accuracy: 0.2500 - val_loss: 1.3863
Epoch 8/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 32ms/step - accuracy: 0.2286 - loss: 1.3864 - val_accuracy: 0.2500 - v

In [73]:
y_pred = model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)

print("Classification Report:\n", classification_report(y_test, y_pred_classes))
print("Accuracy Score: ", accuracy_score(y_test, y_pred_classes))

5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
Classification Report:
               precision    recall  f1-score   support

           0       0.25      1.00      0.40        40
           1       0.00      0.00      0.00        40
           2       0.00      0.00      0.00        40
           3       0.00      0.00      0.00        40

    accuracy                           0.25       160
   macro avg       0.06      0.25      0.10       160
weighted avg       0.06      0.25      0.10       160

Accuracy Score:  0.25


c:\Users\silve\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\silve\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\silve\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, mo

In [74]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.utils.class_weight import compute_class_weight

In [75]:
datagen = ImageDataGenerator(
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

In [76]:
model.fit(
    datagen.flow(X_train, y_train, batch_size=32),  # Use augmented data
    epochs=20,
    validation_data=(X_test, y_test),
)

y_pred = model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)

print("Classification Report:\n", classification_report(y_test, y_pred_classes))
print("Accuracy Score: ", accuracy_score(y_test, y_pred_classes))

Epoch 1/20


c:\Users\silve\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:122: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


20/20 ━━━━━━━━━━━━━━━━━━━━ 3s 65ms/step - accuracy: 0.2381 - loss: 1.3863 - val_accuracy: 0.2500 - val_loss: 1.3863
Epoch 2/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 83ms/step - accuracy: 0.2623 - loss: 1.3863 - val_accuracy: 0.2500 - val_loss: 1.3863
Epoch 3/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 3s 112ms/step - accuracy: 0.2373 - loss: 1.3864 - val_accuracy: 0.2500 - val_loss: 1.3863
Epoch 4/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 3s 126ms/step - accuracy: 0.2771 - loss: 1.3863 - val_accuracy: 0.2500 - val_loss: 1.3863
Epoch 5/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 3s 95ms/step - accuracy: 0.2399 - loss: 1.3864 - val_accuracy: 0.2500 - val_loss: 1.3863
Epoch 6/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 4s 147ms/step - accuracy: 0.2562 - loss: 1.3863 - val_accuracy: 0.2500 - val_loss: 1.3863
Epoch 7/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 76ms/step - accuracy: 0.2435 - loss: 1.3864 - val_accuracy: 0.2500 - val_loss: 1.3863
Epoch 8/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 69ms/step - accuracy: 0.2155 - loss: 1.3864 - val_accuracy: 0.2500 - val_loss:

c:\Users\silve\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\silve\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\silve\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, mo

# more preprocessing

In [ ]:
import imageio
import numpy as np
import sys
sys.path.insert(0, os.path.expanduser("scripts"))
import imgtool as t
import albumentations as A

In [ ]:
def preprocess_image(image_path, target_size=(1300, 52)):
    img = imageio.imread(image_path).astype("float32")
    img = t.pad_or_crop(t.nm(img), target_size)
    return img

In [ ]:
def select_frames(files, num_frames = 15, is_testset = False, test_frame_idx = 0):
    files = files[:num_frames]
    while len(files) < num_frames:  # Duplicate last frame if needed
        files.append(files[-1])
    
    if is_testset:
        return files[test_frame_idx:test_frame_idx + num_frames]
    else:
        start_idx = np.random.randint(len(files) - num_frames + 1)
        return files[start_idx:start_idx + num_frames]

In [ ]:
def augment_images(images, train_aug, test_aug, is_testset):
    transform = A.Compose(test_aug if is_testset else train_aug)
    transformed_images = [transform(image=img)['image'] for img in images]
    return np.stack(transformed_images)


# simplified preporcessing and nn 

In [27]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision.transforms as transforms
import numpy as np
import sys
sys.path.insert(0, os.path.expanduser("scripts"))
from imgtool import ld, rz, nm, makesameshape

In [51]:
def preprocess_image(image_path, target_size=(128, 128)):
    img = ld(image_path)  # Load image as NumPy array
    img = nm(img)  # Normalize pixel values between 0 and 1

    if len(img.shape) == 2:  # If grayscale, add channel dimension
        img = np.expand_dims(img, axis=0)
    return img

In [52]:
def load_dataset(filtered_df_sampled):
    X = []
    y = []
    
    # label mapping
    species_map = {'E. faecalis': 0, 'K. pneumoniae': 1, 'E. coli': 2, 'P. aeruginosa': 3}
    
    for _, row in filtered_df_sampled.iterrows():
        img_path = row['file_path']  # Path to image file
        species_label = row['species_label']  # Target variable
        
        image = preprocess_image(img_path)  # Convert to tensor
        X.append(image)
        
        label = species_map.get(species_label, -1)
        y.append(label)
    
    X = makesameshape(X)  # Ensure consistent shape
    X = np.stack(X, axis=0)  # Convert to NumPy array
    X = torch.tensor(X, dtype=torch.float32)  # Add channel dimension
    
    y = torch.tensor(y, dtype=torch.long)  # Convert labels to tensor
    
    return X, y

In [53]:
X, y = load_dataset(filtered_df_sampled)

In [54]:
import torch.nn as nn

In [55]:
class BacteriaCNN(nn.Module):
    def __init__(self, num_classes=4):
        super(BacteriaCNN, self).__init__()

        self.conv_layers = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
        )

        dummy_input = torch.zeros(1, 1, 128, 128)  # Sample input size
        dummy_output = self.conv_layers(dummy_input)
        flattened_size = dummy_output.view(-1).shape[0]

        self.fc_layers = nn.Sequential(
            nn.Linear(flattened_size, 256),  # Corrected input size
            nn.ReLU(),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = torch.flatten(x, start_dim=1)  # Flatten all dimensions except batch
        x = self.fc_layers(x)
        return x

In [56]:
from torch.utils.data import TensorDataset, DataLoader

In [57]:
batch_size = 32
dataset = TensorDataset(X, y)
train_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

In [58]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BacteriaCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [59]:
epochs = 10
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

    print(f"Epoch {epoch+1}/{epochs}, Loss: {running_loss/len(train_loader):.4f}, Accuracy: {100 * correct/total:.2f}%")

RuntimeError: mat1 and mat2 shapes cannot be multiplied (32x140544 and 32768x256)

In [63]:
img_paths = []
labels = []
species_map = {'E. faecalis': 0, 'K. pneumoniae': 1, 'E. coli': 2, 'P. aeruginosa': 3}
for _, row in filtered_df_sampled.iterrows():
    img_path = row['file_path']  # Path to image file
    species_label = row['species_label']  # Target variable
    
    img_paths.append(img_path)
    labels.append(species_label)

In [ ]:
import os
import numpy as np
import cv2
from skimage.feature import hog
from skimage import exposure
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import classification_report
from sklearn.preprocessing import StandardScaler

# Function to load and preprocess images
def load_image(image_path):
    # Load image
    image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    image = cv2.resize(image, (128, 128))  # Resize to fixed size
    return image

# Function to extract HOG features from image
def extract_hog_features(image):
    # Compute HOG features
    features, hog_image = hog(image, pixels_per_cell=(8, 8), cells_per_block=(2, 2), visualize=True)
    # Normalize the HOG image for display
    hog_image_rescaled = exposure.rescale_intensity(hog_image, in_range=(0, 10))
    return features

# Load dataset paths and labels
def load_dataset(image_paths, labels):
    X = []
    y = []
    for img_path, label in zip(image_paths, labels):
        image = load_image(img_path)
        features = extract_hog_features(image)
        X.append(features)
        y.append(label)
    return np.array(X), np.array(y)

# Example paths and labels
# Replace these with the actual image paths and labels
image_paths = img_paths
labels = labels

# Map the labels to numerical values
species_map = {'E. faecalis': 0, 'K. pneumoniae': 1, 'E. coli': 2, 'P. aeruginosa': 3}
numerical_labels = [species_map[label] for label in labels]

# Split the data into train and test sets
X, y = load_dataset(image_paths, numerical_labels)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale the features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Train a classifier (SVM in this case)
clf = SVC(kernel='linear')
clf.fit(X_train, y_train)

# Make predictions
y_pred = clf.predict(X_test)

# Evaluate the model
print(classification_report(y_test, y_pred))


              precision    recall  f1-score   support

           0       0.96      0.94      0.95        52
           1       0.89      0.93      0.91        55
           2       0.91      0.93      0.92        44
           3       0.91      0.88      0.90        49

    accuracy                           0.92       200
   macro avg       0.92      0.92      0.92       200
weighted avg       0.92      0.92      0.92       200

